# 02 — Feature Discovery

Interactive analysis of Phase 1 results.

Goals:
- Visualize differential activation profiles across layers
- Identify the layer range where override features emerge
- Inspect top features via Neuronpedia autointerpretation
- Classify features into the six-category taxonomy
- Compare feature overlap between CoT unfaithfulness and sycophancy datasets

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PHASE1_RUN = 'FILL_IN_RUN_ID'  # e.g. '20260415_143022'
PHASE1_DIR = Path('../experiments/results/phase1')

In [ ]:
# Load layer summaries for all datasets
datasets = ['cot_bias', 'cot_contradiction', 'sycophancy_opinion', 'sycophancy_pressure', 'cross_domain']
summaries = {}
for ds in datasets:
    p = PHASE1_DIR / PHASE1_RUN / ds / 'layer_summary.json'
    if p.exists():
        with open(p) as f:
            summaries[ds] = json.load(f)
    else:
        print(f'Missing: {p}')

In [ ]:
# Plot: number of significant features per layer, per dataset
fig, axes = plt.subplots(1, len(summaries), figsize=(18, 4), sharey=True)

for ax, (ds, summary) in zip(axes, summaries.items()):
    layers = sorted(int(l) for l in summary.keys())
    n_sig = [summary[str(l)].get('n_significant', 0) for l in layers]
    n_sup = [summary[str(l)].get('n_suppressed', 0) for l in layers]
    n_act = [summary[str(l)].get('n_activated', 0) for l in layers]
    
    ax.bar(layers, n_sup, label='suppressed (delta<0)', color='#d6604d', alpha=0.8)
    ax.bar(layers, n_act, bottom=n_sup, label='activated (delta>0)', color='#2166ac', alpha=0.8)
    ax.set_title(ds, fontsize=9)
    ax.set_xlabel('Layer')
    if ax == axes[0]:
        ax.set_ylabel('# significant features')
    ax.legend(fontsize=7)

plt.suptitle('Significant differential features per layer (Bonferroni α=0.05)', fontsize=11)
plt.tight_layout()
plt.savefig('../paper/figures/phase1_layer_profiles.png', dpi=150)
plt.show()

In [ ]:
# Load differential features for CoT bias and sycophancy pressure
def load_diff_features(ds_name):
    p = PHASE1_DIR / PHASE1_RUN / ds_name / 'differential_features.json'
    with open(p) as f:
        return json.load(f)

cot_diff = load_diff_features('cot_bias')
syco_diff = load_diff_features('sycophancy_pressure')

# Feature overlap: which feature indices appear in both?
cot_feat_set = set()
syco_feat_set = set()
for layer, feats in cot_diff.items():
    cot_feat_set.update((int(layer), f['feature_index']) for f in feats)
for layer, feats in syco_diff.items():
    syco_feat_set.update((int(layer), f['feature_index']) for f in feats)

overlap = cot_feat_set & syco_feat_set
print(f'CoT features: {len(cot_feat_set)}')
print(f'Sycophancy features: {len(syco_feat_set)}')
print(f'Overlap (same layer, same index): {len(overlap)}')
print(f'Overlap fraction (of CoT): {len(overlap)/len(cot_feat_set):.3f}')
print(f'Overlap fraction (of Sycophancy): {len(overlap)/len(syco_feat_set):.3f}')

In [ ]:
# Top features by |delta| across all layers for each condition
def get_top_features(diff_dict, top_k=20):
    all_f = []
    for layer, feats in diff_dict.items():
        for f in feats:
            all_f.append({**f, 'layer': int(layer)})
    return sorted(all_f, key=lambda x: abs(x['delta_activation']), reverse=True)[:top_k]

print('=== Top 10 CoT bias features ===')
for f in get_top_features(cot_diff, 10):
    print(f"  Layer {f['layer']}, Feature {f['feature_index']}: delta={f['delta_activation']:.3f}, effect={f['effect_size']:.3f}")

print('\n=== Top 10 Sycophancy pressure features ===')
for f in get_top_features(syco_diff, 10):
    print(f"  Layer {f['layer']}, Feature {f['feature_index']}: delta={f['delta_activation']:.3f}, effect={f['effect_size']:.3f}")

In [ ]:
# Neuronpedia autointerpretation for top features
# Requires NEURONPEDIA_API_KEY in environment
import os
if os.environ.get('NEURONPEDIA_API_KEY'):
    from src.utils.neuronpedia import NeuronpediaClient
    client = NeuronpediaClient()
    
    # Adjust model_id and sae_id to match Neuronpedia's naming for Gemma Scope 2
    MODEL_NP_ID = 'gemma-3-4b-it'
    SAE_NP_ID = 'FILL_IN_SAE_ID'  # Check Neuronpedia for exact ID
    
    top_cot = get_top_features(cot_diff, 5)
    for f in top_cot:
        try:
            interp = client.autointerp_feature(MODEL_NP_ID, SAE_NP_ID, f['feature_index'])
            print(f"Feature {f['feature_index']} (layer {f['layer']}): {interp.description}")
            print(f"  Autointerp score: {interp.autointerp_score}")
            print(f"  URL: {client.get_circuit_url(MODEL_NP_ID, SAE_NP_ID, f['feature_index'])}")
        except Exception as e:
            print(f"Feature {f['feature_index']}: failed ({e})")
else:
    print('NEURONPEDIA_API_KEY not set. Set it to run autointerpretation.')

In [ ]:
# Effect size distribution: how large are the differences?
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

cot_effects = [f['effect_size'] for feats in cot_diff.values() for f in feats]
syco_effects = [f['effect_size'] for feats in syco_diff.values() for f in feats]

ax1.hist(cot_effects, bins=50, color='#2166ac', alpha=0.8, edgecolor='none')
ax1.set_xlabel("Cohen's d")
ax1.set_ylabel('Count')
ax1.set_title('CoT bias: effect size distribution')
ax1.axvline(np.median(cot_effects), color='red', linestyle='--', label=f'median={np.median(cot_effects):.2f}')
ax1.legend()

ax2.hist(syco_effects, bins=50, color='#d6604d', alpha=0.8, edgecolor='none')
ax2.set_xlabel("Cohen's d")
ax2.set_title('Sycophancy pressure: effect size distribution')
ax2.axvline(np.median(syco_effects), color='blue', linestyle='--', label=f'median={np.median(syco_effects):.2f}')
ax2.legend()

plt.tight_layout()
plt.savefig('../paper/figures/phase1_effect_sizes.png', dpi=150)
plt.show()